# NodeSubstrates - ZINC Molecule Graph

This notebook demonstrates NodeSubstrates on the **ZINC molecule graph** (molecules as nodes, similarity or relation edges).

**Note:** The ZINC dataset file is read from `data/zinc_graph.json`. Nodes may include a SMILES string (`smiles`), a human-readable `name`, computed descriptors or fingerprints (`features`), and optional precomputed embeddings under the `embedding` attribute.

**Use Case:** Explore chemical neighborhoods, cluster molecules by structural similarity or properties, and compare descriptor-based groupings with learned embeddings.

In [5]:
from node_substrates import NodeSubstratesWidget
from node_substrates.datasets.zincmolecules_loader import load_zinc_molecules

## 1. Dataset Overview

Summary

The document includes:

**Node Attributes:**

- `id`: Molecule identifier
- `smiles`: SMILES string describing the molecule structure
- `features`: Descriptor or fingerprint vector (varies by preprocessing)
- `embedding`: Optional precomputed embedding (e.g., Node2Vec or graph-based embedding)
- `property` / `class_name`: Optional molecular property or label if present

**Edge Attributes:**

- `source` & `target`: Relationship between molecules (similarity, scaffold relation, etc.)

**Feature Details:**

- Descriptors or fingerprints can be binary or real-valued depending on preprocessing
- Precomputed embeddings (if present) are suitable for DR and neighborhood comparisons

**Note:** The ZINC JSON was prepared separately; embeddings (if present) were computed offline and saved to the file under the `embedding` attribute.

In [6]:
# Load the ZINC molecule graph
G = load_zinc_molecules()

# nodes_subset = list(G.nodes())[:100]
# G    = G.subgraph(nodes_subset).copy()

print(f"\nNetwork: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Class/property distribution if available
class_counts = {}
for node, attrs in G.nodes(data=True):
    cls = attrs.get('type') or attrs.get('class_name') or attrs.get('category') or 'Unknown'
    class_counts[cls] = class_counts.get(cls, 0) + 1

print("\nProperty/class distribution (if present):")
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls}: {cnt}")

# Sample node attributes (show embedding size if present)
print("\nSample node attributes:")
sample_node = list(G.nodes())[0]
for key, value in G.nodes[sample_node].items():
    if key == 'embedding' and hasattr(value, '__len__'):
        print(f"  {key}: (len={len(value)})")
    else:
        print(f"  {key}: {value}")

Loaded ZINC molecule graph: 3348 nodes, 6912 edges

Network: 3348 nodes, 6912 edges

Property/class distribution (if present):
  atom: 3248
  molecule: 100

Sample node attributes:
  type: molecule
  smiles: C/C=C(\C)[C@@H]1C=C[C@@H]2C[C@H](C)C[C@H](C)[C@@H]2[C@@H]1C(=O)C1=C([O-])[C@H](C[C@](C)(O)C(=O)[O-])NC1=O
  logP: 0.6261
  qed: 0.465834039612
  SAS: 5.692540782615093
  embedding: (len=32)
  label: C/C=C(\C)[C@@H]1C=C[C@@H]2C[C@H](C)C[C@H](C)[C@@H]2[C@@H]1C(=O)C1=C([O-])[C@H](C[C@](C)(O)C(=O)[O-])NC1=O
  degree: 32
  clustering: 0.0685
  betweenness: 8.8e-05


## 2. Create NodeSubstrates Widget

The initial view shows a force-directed layout revealing molecule neighborhoods and scaffold communities.

Use the hybrid view to compare fingerprint/descriptor features, structural embeddings (if present), and graph-derived metrics (degree, betweenness).

In [7]:
# Ensure histograms show LogP, SAS, and QED together in preferred order
preferred = ['logP', 'SAS', 'qed']
prefix_fmt = '{:02d} - {}'
mappings = {k: prefix_fmt.format(i+1, k) for i,k in enumerate(preferred)}



print('Applying histogram key mapping for order:')
for k,v in mappings.items():
    print(f"  {v} <= {k}")

# Duplicate attributes onto each node with prefixed keys (keeps originals intact)
for n, attrs in G.nodes(data=True):
    for orig_key, new_key in mappings.items():
        if orig_key in attrs:
            # copy value under the new prefixed key
            attrs[new_key] = attrs[orig_key]

print('Done. Create the widget next (as in the notebook).')

Applying histogram key mapping for order:
  01 - logP <= logP
  02 - SAS <= SAS
  03 - qed <= qed
Done. Create the widget next (as in the notebook).


In [8]:
# Create widget with zoomed out view for better layout spread
# initial_scale=0.3 computes layout for ~3x larger area in each dimension
widget = NodeSubstratesWidget(G, auto_substrate=False, width=1700, height=600, initial_scale=0.60)

# Interaction hints:
# - Pan: drag on empty space
# - Zoom: scroll wheel
# - Lasso select: Shift+drag (popup appears automatically)
# - Move substrate: Ctrl+drag on substrate region
# - Context menu: right-click on node or substrate
widget

## 4. Create a Substrate for High-Connectivity Molecules

Let's create a substrate focusing on the most connected molecules to examine scaffold-driven clusters and hub compounds.

In [ ]:
# Select high-degree molecules (hub compounds)
high_degree = [
    str(node) for node, attrs in G.nodes(data=True)
    if attrs.get('degree', 0) >= 10
]

print(f"Found {len(high_degree)} high-degree molecules")

if len(high_degree) >= 3:
    substrate_id = widget.create_substrate(
        high_degree,
        dr_method='umap',
        label='High-Degree Molecules'
    )
    print(f"Created substrate: {substrate_id}")

## 5. Compare DR Methods

Different dimensionality reduction methods reveal different patterns:
- **PCA**: Linear relationships in descriptor space
- **UMAP**: Non-linear scaffold or property clusters
- **t-SNE**: Local neighborhood emphasis

Compare fingerprint/descriptor DR with learned embeddings (if present) to
see whether structural embeddings capture different relationships than
hand-crafted descriptors.

In [ ]:
# Try UMAP for cluster discovery
if widget.substrates:
    widget.update_dr_method(widget.substrates[0]['id'], 'umap')
    print("Switched to UMAP - look for clusters of similar behavior patterns")

## 6. Interactive Exploration

Use lasso selection and substrates to investigate molecule neighborhoods:

- **Shift+Drag** to select nodes
- Create substrates from selections to compare scaffold neighborhoods
- Compare descriptor-based clusters vs. embedding neighborhoods

In [ ]:
# Check current selection
print(f"Selected nodes: {widget.selected_nodes}")

# Create substrate from selection (need at least 3 nodes)
if len(widget.selected_nodes) >= 3:
    substrate_id = widget.create_substrate(
        widget.selected_nodes,
        dr_method='umap',
        label='Investigation Selection'
    )
    print(f"Created substrate from selection: {substrate_id}")

## 7. Current Substrates

In [ ]:
# List current substrates
print("Current substrates:")
for s in widget.substrates:
    print(f"  {s['id']}: {s['label']} ({len(s['node_ids'])} nodes, {s['dr_method']})")

In [ ]:
# Dissolve a substrate to return nodes to force-directed layout
if widget.substrates:
    substrate_to_dissolve = widget.substrates[0]['id']
    widget.dissolve_substrate(substrate_to_dissolve)
    print(f"Dissolved {substrate_to_dissolve}")

## 8. ZINC Graph Insights

The hybrid NodeSubstrates view helps chemists and data scientists by:

**Network View (Force-Directed):**
- Reveals scaffold communities and similarity neighborhoods
- Shows hub molecules and bridging compounds between scaffold families

**Attribute / Embedding View (Substrates):**
- Groups molecules by descriptors or learned embeddings
- Highlights outliers with unusual structural signatures
- Enables side-by-side comparison of descriptor neighborhoods and embedding neighborhoods

**Key analysis points:**
- Hub compounds that may be central to a scaffold family
- Molecules bridging different scaffold groups
- Differences between fingerprint-based clusters and learned embeddings

In [ ]:
# Summary statistics
print(f"\n=== ZINC Graph Summary ===")
print(f"Total nodes: {len(widget.nodes)}")
print(f"Nodes in substrates: {len(widget.substrate_node_ids)}")
print(f"Nodes in force-directed: {len(widget.topological_node_ids)}")
print(f"Active substrates: {len(widget.substrates)}")

# Property/class breakdown (if any)
print(f"\n=== Property / Class Distribution ===")
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls}: {count}")